# What transfers? Readable equations and a ledger of what holds on new cases

The dataset notebooks in this series found the same pattern again and again: a model that
scores well when test reactions reuse molecules it was trained on can fail on a molecule it
has never seen. This notebook makes that systematic with `transferledger`, a small package
that lives in this repository (`packages/transferledger`) and knows nothing about chemistry:
it takes a table, a target and a statement of what "new" means.

**Readable equations.** A model that is a sum of one curve per variable (penalized splines by
default; a KAN with no hidden layer, `layers=(d, 1)`, is the same family and is used below as a
second source) gives one curve per descriptor. Each curve is matched to the simplest named form
that reproduces it (a line, 1/x, an exponential, a quadratic with its optimum, a V, a step, a
window), with one support rule: an edge needs at least three distinct values on each side,
otherwise it is a lookup of a few molecules and not a shape. Each matched form is a *piece*, in
the descriptor's own units. The equation is the sum of the pieces that earn their place on
held-out rows, plus products of two pieces where one variable changes the effect of another.

**Shifts.** "Does this transfer?" has no answer until one says to what. A shift is a way of
holding cases out: whole groups (a new ligand, a new alcohol), a range beyond training (a higher
temperature), a region of descriptor space not trained on. Several are given at once, and a
verdict always names its shift.

**The ledger.** Every model is scored on held-out rows under the *reference* (random folds:
test cases resemble training cases) and under each shift. Skill is the share of the baseline's
loss removed, where the baseline predicts the training mean: 0 is knowing nothing, negative is
worse than that. Each number has an interval from leaving out one unit at a time (a whole
molecule, when molecules are held out), so four ligands give wide intervals. The rungs:

- *equation*: pieces chosen on random folds;
- *equation, chosen for a shift*: a variable is admitted only when it improves predictions
  under that shift, by more than its own uncertainty over the shift's units;
- *curves*: the additive model the pieces came from, also tuned for each shift;
- the usual one-hidden-layer KAN of the dataset notebooks, and gradient boosting.

Each variable of the equation is then removed in turn, and the skill lost is recorded under the
reference and under each shift. A variable *transfers* when it helps under both, *does not
transfer* when it helps on the reference and hurts under the shift, and is *undetermined* when
the shift's interval spans zero. Descriptors of one component are also removed together, as a
block, because near-duplicates hide each other one at a time.

**The memorized share** is the reference skill an equation gives up when its variables must
earn their place under the shift: what it knew only by having seen the cases.

Section 1 is a calibration with a known answer, including an interaction. Sections 2 to 6 are
the five datasets, each loaded as in its own notebook, with every component held out in turn.
Section 7 is a dataset from another field, with the same calls. All data are downloaded at run
time. The package is installed with `pip install -e packages/transferledger`.

In [ ]:
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"
import time
import warnings

warnings.filterwarnings("ignore")

import importlib.metadata as md
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

from pycse.sklearn._plotstyle import BASELINE, MUTED, PX, SERIES, finish_axes, style_axes
from pycse.sklearn.kan import KAN
from pycse.sklearn.kan_interpret import prune_collinear
from transferledger import BeyondRange, NewGroups, PieceModel, transfer_ledger

HAS_PYSR = importlib.util.find_spec("pysr") is not None
versions = {
    p: md.version(p) for p in ["transferledger", "pycse", "jax", "numpy", "scikit-learn", "scipy"]
}
if HAS_PYSR:
    versions["pysr"] = md.version("pysr")
print("Versions: " + ", ".join(f"{k} {v}" for k, v in versions.items()))
N_JOBS = min(4, os.cpu_count() or 1)
# set LEDGER_OUT to a folder to also get each ledger as an HTML report and as JSON tables
LEDGER_OUT = os.environ.get("LEDGER_OUT")
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 120)

## Helpers

In [ ]:
LEDGERS = {}
COLORS = [MUTED, *SERIES, "#8a5cd0", "#b8860b"]


class KANRung(KAN):
    # the one-hidden-layer KAN of the dataset notebooks, with their iteration limit
    def fit(self, X, y):
        return super().fit(X, y, maxiter=300)


def upper_rungs(n_rows, n_features, family):
    wide = n_rows >= 1000
    kan = KANRung(layers=(n_features, 8 if wide else 4, 1), grid_size=5 if wide else 3,
                  l1_spline=1e-3, entropy_reg=1e-3)  # fmt: skip
    gb = (GradientBoostingRegressor if family == "gaussian" else GradientBoostingClassifier)(
        n_estimators=300, learning_rate=0.1, max_depth=4, min_samples_split=5,
        min_samples_leaf=3, subsample=0.8, random_state=42,
    )  # fmt: skip
    return {"KAN, one hidden layer": kan, "gradient boosting": gb}


def whiskers(ax, table, rows, schemes, floor):
    # one dot and interval per row and scheme; values below the floor are written out
    pos = np.arange(len(rows))
    step = 0.7 / max(len(schemes), 1)
    for k, s in enumerate(schemes):
        t = table[table.scheme == s].set_index(table.columns[0]).reindex(rows)
        v, lo, hi = (t[c].to_numpy(float) for c in t.columns[1:4])
        off = pos + (k - (len(schemes) - 1) / 2) * step
        ok = np.isfinite(v)
        ax.hlines(off[ok], np.maximum(lo[ok], floor), np.maximum(hi[ok], floor),
                  color=COLORS[k % len(COLORS)], linewidth=1.5)  # fmt: skip
        ax.plot(np.maximum(v[ok], floor), off[ok], "o", color=COLORS[k % len(COLORS)],
                markersize=4.5, label=s)  # fmt: skip
        for p, val in zip(off[ok], v[ok]):
            if val < floor:
                ax.text(floor, p, f"  {val:.1f}", va="center", fontsize=7)
    ax.set_yticks(pos)
    ax.set_yticklabels(rows, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0.0, color=BASELINE, linewidth=PX, linestyle="--")


def plot_ledger(L, title):
    rungs = [r for r in L.ladder.index if not r.startswith("baseline")]
    parts = L.blocks if len(L.blocks) else L.contributions
    label = "block" if len(L.blocks) else "variable"
    rows = list(dict.fromkeys(parts[label]))[:12]
    height = max(3.6, 0.42 * max(len(rungs), len(rows)) + 1.2)
    fig, axes = plt.subplots(1, 2, figsize=(13, height), gridspec_kw={"width_ratios": [1.25, 1]})
    style_axes(axes[0])
    whiskers(axes[0], L.skill, rungs, L.schemes, -1.0)
    axes[0].set_xlim(-1.0, 1.0)
    axes[0].legend(frameon=False, fontsize=8, loc="lower left")
    finish_axes(
        axes[0], f"{title}: skill of each rung", "skill (share of the baseline's loss removed)", ""
    )
    style_axes(axes[1])
    lo = min(-0.2, float(np.nanmin(parts.set_index(label).loc[rows, "lo"].clip(lower=-1.0))) - 0.05)
    whiskers(axes[1], parts, rows, L.schemes, lo)
    what = "descriptor blocks" if len(L.blocks) else "variables"
    finish_axes(axes[1], f"What the {what} add", "skill lost when removed", "")
    plt.tight_layout()
    plt.show()


def run_dataset(tag, X_df, y, shifts, *, family="gaussian", component=None, forms=None,
                hypothesis=(), prune=True, kan=True):  # fmt: skip
    names = list(X_df.columns)
    if prune:
        kept, dropped = prune_collinear(
            X_df.to_numpy(dtype=float), names, 0.85, keep_first=list(hypothesis)
        )
        for n, (partner, r) in dropped.items():
            print(f"pruned {n}: |r| = {r:.2f} with {partner}")
        names = kept
    X = X_df[names].reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    blocks = {}
    for n in names:
        if component and n in component:
            blocks.setdefault(component[n], []).append(n)
    blocks = {b: cols for b, cols in blocks.items() if len(cols) > 1 or len(blocks) > 1}
    rungs = upper_rungs(len(X), len(names), family)
    if not kan:
        rungs.pop("KAN, one hidden layer")
    print(f"{tag}: {len(X)} rows, {len(names)} variables, family {family}; shifts: "
          + ", ".join(f"{k} ({len(np.unique(s.units()))} units)" for k, s in shifts.items()))  # fmt: skip
    t0 = time.time()
    L = transfer_ledger(X, y, shifts, model=PieceModel(family=family, forms=forms), family=family,
                        blocks=blocks, models=rungs, model_repeats=1, n_jobs=N_JOBS)  # fmt: skip
    print(f"({time.time() - t0:.0f} s)\n")
    for w in L.preflight.warnings:
        print("pre-check:", w)
    print(L)
    plot_ledger(L, tag)
    for name, m in L.models.items():
        print(f"\n{name} (fitted to all {len(X)} rows; held-out skill {m.skill_:.2f}):")
        print(m.equation())
    if LEDGER_OUT:
        stem = os.path.join(LEDGER_OUT, tag.lower().replace(" ", "_").replace(",", ""))
        L.report(stem + ".html", title=f"Transfer ledger: {tag}")
        L.to_json(stem + ".json")
    LEDGERS[tag] = L
    return L


def groups_of(frame, columns):
    # one group per distinct row of the given columns (the identity of a component)
    return frame[list(columns)].round(6).astype(str).agg("|".join, axis=1).to_numpy()

## 1. Calibration: a known equation with an interaction

Six hundred reactions generated from textbook terms, as in `KAN_Known_Truth_Benchmark`, except
that the Hammett slope changes across the ligation cliff: rho = +2.0 for ligands below 32 %
buried volume and -0.5 above it, a change of mechanism with the ligation state.

log10 k = -3 + (2.0 - 2.5 S) sigma - 1.5 S - 15/(2.303 R) (1/T - 1/298.15) - 0.01 (dE + 15)^2,
with S = 1 / (1 + exp(-(%Vbur - 32))), plus an unrelated descriptor and noise of 0.15.

What a correct analysis returns: a step at 32 %, a 1/T term whose weight gives 15 kcal/mol, an
optimum at -15, a Hammett line whose slope is +2.0 below the step and -0.5 above it (the
product of the line with the step), and the unrelated descriptor left out. Two shifts are
asked: a substituent never seen, and temperatures above every training temperature. Nothing
here can be memorized, so every rung should keep its skill on new substituents; a 1/T term
should carry to higher temperatures.

In [ ]:
R_CAL = 1.987e-3
SIGMA_P = {"NO2": 0.78, "CN": 0.66, "CF3": 0.54, "CO2Me": 0.45, "Cl": 0.23, "Br": 0.23,
           "F": 0.06, "H": 0.00, "Me": -0.17, "tBu": -0.20, "OMe": -0.27, "NMe2": -0.83}  # fmt: skip
rng = np.random.default_rng(2021)
N = 600
subst = rng.choice(list(SIGMA_P), size=N)
cal = pd.DataFrame({
    "sigma_p": [SIGMA_P[s] for s in subst], "T_K": rng.uniform(288.0, 368.0, N),
    "pct_Vbur_min": rng.uniform(22.0, 50.0, N), "dE_bind": rng.uniform(-30.0, 0.0, N),
    "noise_descriptor": rng.normal(size=N),
})  # fmt: skip
S_cliff = 1.0 / (1.0 + np.exp(-(cal.pct_Vbur_min - 32.0)))
y_cal = (-3.0 + (2.0 - 2.5 * S_cliff) * cal.sigma_p - 1.5 * S_cliff
         - 15.0 / (2.303 * R_CAL) * (1.0 / cal.T_K - 1.0 / 298.15)
         - 0.01 * (cal.dE_bind + 15.0) ** 2 + rng.normal(0, 0.15, N)).to_numpy()  # fmt: skip
CAL_FORMS = {"T_K": ["inverse"]}  # the field names this relationship: an Arrhenius term
cal_shifts = {
    "new substituent": NewGroups(subst, "new substituent"),
    "higher temperature": BeyondRange(cal.T_K.to_numpy(), "high", name="higher temperature"),
}
L = run_dataset("Calibration", cal, y_cal, cal_shifts, forms=CAL_FORMS, prune=False)

In [ ]:
def recovered(model, X):
    # the generating constants, read back from the equation's terms
    t = model.terms_.set_index("variable")
    out = {}
    if "pct_Vbur_min" in t.index:
        out["step in %Vbur at (generated: 32)"] = t.loc["pct_Vbur_min", "location"]
    if "dE_bind" in t.index:
        out["optimum in dE at (generated: -15)"] = t.loc["dE_bind", "location"]
    if "T_K" in t.index:  # the piece is mean(T) / T
        out["barrier, kcal/mol (generated: 15)"] = (
            -t.loc["T_K", "weight"] * X.T_K.mean() * 2.303 * R_CAL
        )
    prod = [v for v in t.index if " x " in v and "sigma_p" in v and "pct_Vbur_min" in v]
    if "sigma_p" in t.index:
        sd = X.sigma_p.std(ddof=0)
        out["Hammett slope below the step (generated: +2.0)"] = t.loc["sigma_p", "weight"] / sd
        if prod:
            out["Hammett slope above the step (generated: -0.5)"] = (
                t.loc["sigma_p", "weight"] + t.loc[prod[0], "weight"]) / sd  # fmt: skip
    out["unrelated descriptor in the equation"] = "noise_descriptor" in model.variables_
    return pd.Series(out)


spline_model = L.models["equation"]
t0 = time.time()
kan_model = PieceModel(source="kan", forms=CAL_FORMS).fit(cal, y_cal)
print(f"The same call with an additive KAN as the curve source ({time.time() - t0:.0f} s):")
print(kan_model.equation())
display(
    pd.DataFrame(
        {"from splines": recovered(spline_model, cal), "from a KAN": recovered(kan_model, cal)}
    )
)
if HAS_PYSR:
    t0 = time.time()
    sr_model = PieceModel(products="pysr", forms=CAL_FORMS, pysr_options={"niterations": 40}).fit(
        cal, y_cal
    )
    print(f"\nWith PySR assembling the pieces ({time.time() - t0:.0f} s); held-out skill {sr_model.skill_:.3f} "
          f"against {spline_model.skill_:.3f} for the built-in products:")  # fmt: skip
    print(sr_model.equation())

## 2. Known-truth benchmark (additive)

The synthetic data of `KAN_Known_Truth_Benchmark`: the same terms with no interaction. The equation should come back term by term, no product should be invented, and everything should transfer to a new substituent.

In [ ]:
R_CAL = 1.987e-3  # kcal/(mol K)
SIGMA_P = {
    "NO2": 0.78, "CN": 0.66, "CF3": 0.54, "CO2Me": 0.45, "Cl": 0.23, "Br": 0.23,
    "F": 0.06, "H": 0.00, "Me": -0.17, "tBu": -0.20, "OMe": -0.27, "NMe2": -0.83,
}  # fmt: skip
RHO, DH_ACT, CLIFF_DROP, CLIFF_W, VOLCANO_A, E_OPT = 1.5, 15.0, 1.5, 1.0, 0.01, -15.0
LOG_K0, NOISE_SD, N = -3.0, 0.15, 600


def term_hammett(s):
    return RHO * s


def term_eyring(T):
    return -DH_ACT / (2.303 * R_CAL) * (1.0 / T - 1.0 / 298.15)


def term_cliff(v):
    return -CLIFF_DROP / (1.0 + np.exp(-(v - 32.0) / CLIFF_W))


def term_volcano(e):
    return -VOLCANO_A * (e - E_OPT) ** 2


def term_none(z):
    return np.zeros_like(np.asarray(z, dtype=float))


TRUTH = {
    "sigma_p": term_hammett,
    "T_K": term_eyring,
    "pct_Vbur_min": term_cliff,
    "dE_bind": term_volcano,
    "noise_descriptor": term_none,
}

rng = np.random.default_rng(2021)
subst = rng.choice(list(SIGMA_P), size=N)
df = pd.DataFrame(
    {
        "substituent": subst,
        "sigma_p": [SIGMA_P[s] for s in subst],
        "T_K": rng.uniform(288.0, 368.0, N),
        "pct_Vbur_min": rng.uniform(22.0, 50.0, N),
        "dE_bind": rng.uniform(-30.0, 0.0, N),
        "noise_descriptor": rng.normal(size=N),
    }
)
df["log10_k"] = LOG_K0 + sum(TRUTH[f](df[f].to_numpy()) for f in TRUTH) + rng.normal(0, NOISE_SD, N)

X_df = df[list(TRUTH)]
y = df["log10_k"].to_numpy()
TASK = "regression"
Y_LABEL = "log10 k"
COMPONENT = {
    "sigma_p": "substrate",
    "T_K": "conditions",
    "pct_Vbur_min": "ligand",
    "dE_bind": "catalyst",
    "noise_descriptor": "none",
}
SUCCESS = (y > np.median(y)).astype(int)
SUCCESS_RULE = "log10 k above the median"
_ = df.head()
GROUPS = df["substituent"].to_numpy()
HYPOTHESIS = ["sigma_p", "T_K", "pct_Vbur_min", "dE_bind"]

In [ ]:
shifts = {"new substituent": NewGroups(GROUPS, "new substituent")}
L = run_dataset(
    "Known-truth benchmark (additive)",
    X_df,
    y,
    shifts,
    family="gaussian",
    forms={"T_K": ["inverse"]},
    prune=False,
)

## 3. Phosphine ligation-state cliff

Newman-Stonebraker et al., Science 2021: 450 Ni-catalysed couplings over 90 phosphines, active or not (a yes/no target: the equation is for the log-odds). Two shifts: a ligand never seen, and a *kind* of ligand never seen (ligands clustered by their descriptors, whole clusters held out), which is the harder and more useful question.

In [ ]:
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")  # the kraken table has blank SMILES rows

BASE = "https://raw.githubusercontent.com/doyle-lab-ucla/ochem-data/main/"
rxn = pd.read_csv(BASE + "Vbur/rxns/smiles.csv")
kraken = pd.read_csv(BASE + "kraken/kraken.csv")
SPHERE = 4.0 / 3.0 * np.pi * 3.5**3  # 179.6 cubic angstrom


def canon(s):
    m = Chem.MolFromSmiles(str(s))
    return Chem.MolToSmiles(m) if m is not None else None


kraken["key"] = kraken["ligand"].astype(str).str.lower().str.strip()
kraken["csmi"] = kraken["can_smiles"].where(kraken["can_smiles"].notna()).map(canon)
kraken = kraken.drop_duplicates("key")
rxn["key"] = rxn["ligand_name"].str.lower().str.strip()
rxn["csmi"] = rxn["ligand_smiles"].map(canon)

by_name = rxn.merge(kraken, on="key", how="left", suffixes=("", "_k"))
missing = by_name["id"].isna()
by_smi = rxn[missing].merge(
    kraken.drop_duplicates("csmi"), on="csmi", how="left", suffixes=("", "_k")
)
data = pd.concat([by_name[~missing], by_smi]).reset_index(drop=True)
print(
    f"{len(rxn)} reactions; matched by name {(~missing).sum()}, by SMILES {by_smi['id'].notna().sum()}, unmatched {data['id'].isna().sum()}"
)
data = data[data["id"].notna()].copy()

KRAKEN = {
    "pct_Vbur_min": ("vbur_vbur_min", 100.0 / SPHERE),
    "pct_Vbur_Boltz": ("vbur_vbur_boltz", 100.0 / SPHERE),
    "pct_Vbur_max": ("vbur_vbur_max", 100.0 / SPHERE),
    "Sterimol_B1": ("sterimol_B1_boltz", 1.0),
    "Sterimol_B5": ("sterimol_B5_boltz", 1.0),
    "Sterimol_L": ("sterimol_L_boltz", 1.0),
    "E_HOMO": ("fmo_e_homo_boltz", 1.0),
    "E_LUMO": ("fmo_e_lumo_boltz", 1.0),
    "V_min": ("vmin_vmin_boltz", 1.0),
    "NBO_charge_P": ("nbo_P_boltz", 1.0),
    "pyramidalization_P": ("pyr_P_boltz", 1.0),
    "dipole": ("dipolemoment_boltz", 1.0),
}
X_df = pd.DataFrame({k: data[c].astype(float) * s for k, (c, s) in KRAKEN.items()})

# the five substrate pairs, by SMILES
ELEC = {"ClC1=CC=C(C(F)(F)F)C=C1": "4-CF3-C6H4Cl", "ClC1=CC=C(OC)C=C1": "4-MeO-C6H4Cl"}
NUC = {
    "OB(O)C1=CC=C(C(F)(F)F)C=C1": "4-CF3-phenyl", "OB(O)C1=C(C)C=C(C)C=C1C": "mesityl",
    "OB(O)C1=CC=C(C(OCC)=O)C=C1": "4-CO2Et-phenyl", "OB(O)C1=CC=C(OC)C=C1": "4-MeO-phenyl",
    "OB(O)C1=CC=CC=C1": "phenyl",
}  # fmt: skip
data["reaction"] = (
    data.chloride_smiles.map(ELEC) + " + " + data.boronic_acid_smiles.map(NUC) + "B(OH)2"
)
assert data["reaction"].notna().all() and data["reaction"].nunique() == 5
REACTIONS = sorted(data["reaction"].unique())
for r in REACTIONS[1:]:  # the first pair is the reference level
    X_df["rxn: " + r] = (data["reaction"] == r).astype(float)
X_df.index = data.index

# Active, relative to the reaction. The five pairs differ in intrinsic reactivity: the best
# ligand reaches 91 % with one pair and 25 % with another, so one pooled yield cut would call
# most ligands active in the easy pair and almost none in the hard one. The paper shades an
# inactive band per reaction; here a ligand counts as active when it reaches a quarter of the
# best yield obtained with that same substrate pair.
best = data.groupby("reaction")["yield"].transform("max")
ACTIVE_FRACTION = 0.25
y = (data["yield"] >= ACTIVE_FRACTION * best).astype(int).to_numpy()
TASK = "classification"
Y_LABEL = "P(active)"
SUCCESS = y
SUCCESS_RULE = (
    f"yield at least {ACTIVE_FRACTION:.0%} of the best yield with the same substrate pair"
)
COMPONENT = {
    k: "ligand sterics"
    for k in [
        "pct_Vbur_min",
        "pct_Vbur_Boltz",
        "pct_Vbur_max",
        "Sterimol_B1",
        "Sterimol_B5",
        "Sterimol_L",
    ]
}
COMPONENT.update(
    {
        k: "ligand electronics"
        for k in ["E_HOMO", "E_LUMO", "V_min", "NBO_charge_P", "pyramidalization_P", "dipole"]
    }
)
COMPONENT.update({c: "substrate pair" for c in X_df.columns if c.startswith("rxn:")})
print(
    data.groupby("reaction")
    .agg(
        mean_yield=("yield", "mean"),
        best=("yield", "max"),
        n=("yield", "size"),
        active=("yield", lambda s: (s >= ACTIVE_FRACTION * s.max()).mean()),
    )
    .round(2)
)
_ = X_df.describe().T[["mean", "min", "max"]].round(2)
GROUPS = data["key"].to_numpy()
HYPOTHESIS = ["pct_Vbur_min"]

In [ ]:
lig_cols = [c for c in X_df.columns if not c.startswith("rxn:")]
shifts = {
    "new ligand": NewGroups(GROUPS, "new ligand"),
    "new kind of ligand": NewGroups.from_clusters(
        X_df[lig_cols].to_numpy(float), 8, "new kind of ligand"
    ),
}
L = run_dataset(
    "Phosphine ligation-state cliff",
    X_df,
    y,
    shifts,
    family="binomial",
    component=COMPONENT,
    hypothesis=HYPOTHESIS,
)

## 4. Buchwald-Hartwig amination, DFT descriptors

Ahneman et al., Science 2018: 3955 reactions over 15 aryl halides, 22 isoxazole additives, 3 bases and 4 ligands. Each component is held out in turn. With 3 bases and 4 ligands the groups are left out one at a time and the intervals are wide: that is what three bases can support.

In [ ]:
URL = "https://raw.githubusercontent.com/b-shields/edbo/master/examples/DOE/data/aryl_amination/science_dft.csv"
RXN = "https://raw.githubusercontent.com/doylelab/rxnpredict/master/"
raw = pd.read_csv(URL)
CHOSEN = {
    "additive": [
        "additive_E_LUMO", "additive_E_HOMO", "additive_.C3_NMR_shift", "additive_.C4_NMR_shift",
        "additive_.C5_NMR_shift", "additive_.N1_electrostatic_charge", "additive_.O1_electrostatic_charge",
        "additive_dipole_moment", "additive_molecular_volume",
    ],
    "aryl halide": [
        "aryl_halide_E_LUMO", "aryl_halide_E_HOMO", "aryl_halide_.C1_NMR_shift",
        "aryl_halide_.C1_electrostatic_charge", "aryl_halide_.C4_NMR_shift", "aryl_halide_.H2_NMR_shift",
        "aryl_halide_dipole_moment", "aryl_halide_molecular_volume", "aryl_halide_hardness",
    ],
    "base": ["base_E_HOMO", "base_.N1_electrostatic_charge", "base_molecular_volume"],
    "ligand": ["ligand_.P1_electrostatic_charge", "ligand_dipole_moment", "ligand_.C1_NMR_shift"],
}  # fmt: skip
COMPONENT = {c: comp for comp, cols in CHOSEN.items() for c in cols}
X_df = raw[list(COMPONENT)].copy()
const = [c for c in X_df.columns if X_df[c].nunique() <= 1]
X_df = X_df.drop(columns=const)
print(f"Dropped constant descriptors: {const}")
y = raw["yield"].to_numpy(dtype=float)
TASK = "regression"
Y_LABEL = "yield (%)"
SUCCESS = (y > 20.0).astype(int)
SUCCESS_RULE = "yield above 20 %"

stats = X_df.agg(["mean", "std"]).T
print(f"Descriptor means are {stats['mean'].abs().max():.3f} or less and standard deviations "
      f"{stats['std'].min():.3f} to {stats['std'].max():.3f}: the table is standardized.")  # fmt: skip


def recover_names(key_col, table_col, table, names, checks):
    """Map each dataset level of key_col to a molecule name, then verify on other descriptors."""
    rank = np.argsort(np.argsort(table[table_col].to_numpy()))
    levels = np.sort(raw[key_col].unique())
    assert len(levels) == len(table), f"{len(levels)} levels vs {len(table)} molecules"
    mapping = {i: levels[rank[i]] for i in range(len(table))}
    for s_col, t_col in checks:
        lv, rk = np.sort(raw[s_col].unique()), np.argsort(np.argsort(table[t_col].to_numpy()))
        for i in range(len(table)):
            seen = raw.loc[raw[key_col] == mapping[i], s_col].unique()
            assert len(seen) == 1 and np.isclose(seen[0], lv[rk[i]]), (
                f"{t_col} disagrees for {names[i]}"
            )
    return {mapping[i]: names[i] for i in range(len(table))}


def name_list(stem):
    n = pd.read_csv(RXN + f"smiles/{stem}-list.csv")
    return n["name"].tolist()[: len(pd.read_csv(RXN + f"R/{stem}.csv"))]


tables = {s: pd.read_csv(RXN + f"R/{s}.csv") for s in ["aryl_halide", "additive", "base", "ligand"]}
raw["aryl_halide"] = raw["aryl_halide_.C1_NMR_shift"].map(
    recover_names("aryl_halide_.C1_NMR_shift", "aryl_halide_*C1_NMR_shift", tables["aryl_halide"], name_list("aryl_halide"),
                  [("aryl_halide_.C4_NMR_shift", "aryl_halide_*C4_NMR_shift"), ("aryl_halide_E_LUMO", "aryl_halide_E_LUMO"),
                   ("aryl_halide_molecular_volume", "aryl_halide_molecular_volume")])
)  # fmt: skip
raw["additive"] = raw["additive_.C3_NMR_shift"].map(
    recover_names("additive_.C3_NMR_shift", "additive_*C3_NMR_shift", tables["additive"], name_list("additive"),
                  [("additive_.C4_NMR_shift", "additive_*C4_NMR_shift"), ("additive_E_LUMO", "additive_E_LUMO")])
)  # fmt: skip
raw["base"] = raw["base_E_HOMO"].map(
    recover_names("base_E_HOMO", "base_E_HOMO", tables["base"], name_list("base"),
                  [("base_molecular_volume", "base_molecular_volume"), ("base_.N1_electrostatic_charge", "base_*N1_electrostatic_charge")])
)  # fmt: skip
raw["ligand"] = raw["ligand_dipole_moment"].map(
    recover_names("ligand_dipole_moment", "ligand_dipole_moment", tables["ligand"], name_list("ligand"),
                  [("ligand_.C1_NMR_shift", "ligand_*C1_NMR_shift"), ("ligand_.P1_electrostatic_charge", "ligand_*P1_electrostatic_charge")])
)  # fmt: skip
raw["halide"] = np.select(
    [raw.aryl_halide.str.contains("iodo"), raw.aryl_halide.str.contains("bromo")], ["I", "Br"], "Cl"
)

# The scaffold is the aryl halide without its halogen, keeping where the halogen was:
# 2- and 3-halopyridines are different substrates even though both lose a halogen to
# give pyridine. The names are written inconsistently ("1-ethyl-4-iodobenzene" but
# "1-bromo-4-ethylbenzene"), so the scaffold is read from the substituent named.


def scaffold_of(name):
    if "pyridine" in name:
        return f"pyridin-{name[0]}-yl"
    for key, label in [
        ("trifluoromethyl", "4-(trifluoromethyl)phenyl"),
        ("methoxy", "4-methoxyphenyl"),
        ("ethyl", "4-ethylphenyl"),
    ]:
        if key in name:
            return label
    raise ValueError(name)


raw["scaffold"] = raw.aryl_halide.map(scaffold_of)
assert raw["scaffold"].nunique() == 5
print("Names recovered and verified against two further descriptors each.")
print(f"{len(raw)} reactions: {raw.aryl_halide.nunique()} aryl halides, {raw.additive.nunique()} additives, "
      f"{raw.base.nunique()} bases ({', '.join(sorted(raw.base.unique()))}), {raw.ligand.nunique()} ligands "
      f"({', '.join(sorted(raw.ligand.unique()))}).")  # fmt: skip
GROUPS = raw["additive"].to_numpy()
HYPOTHESIS = []

In [ ]:
shifts = {
    f"new {c.replace('_', ' ')}": NewGroups(raw[c].to_numpy(), f"new {c.replace('_', ' ')}")
    for c in ["additive", "aryl_halide", "base", "ligand"]
}
L = run_dataset(
    "Buchwald-Hartwig amination, DFT descriptors",
    X_df,
    y,
    shifts,
    family="gaussian",
    component=COMPONENT,
)

## 5. Direct arylation with reaction conditions

Shields et al., Nature 2021: 1728 reactions, a full factorial over 12 ligands, 4 bases, 4 solvents, 3 concentrations and 3 temperatures. Held out in turn: ligands, bases, solvents, and the highest temperature.

In [ ]:
BASE = "https://raw.githubusercontent.com/b-shields/edbo/master/examples/DOE/data/direct_arylation/"
idx = pd.read_csv(BASE + "experiment_index.csv")
lig = pd.read_csv(BASE + "ligand-boltzmann_dft.csv")
bas = pd.read_csv(BASE + "base_dft.csv")
sol = pd.read_csv(BASE + "solvent_dft.csv")

LIG = {
    "ligand_pct_Vbur_P": "ligand_atom1_%VBur_Boltz", "ligand_E_HOMO": "ligand_homo_energy_Boltz",
    "ligand_E_LUMO": "ligand_lumo_energy_Boltz", "ligand_hardness": "ligand_hardness_Boltz",
    "ligand_dipole": "ligand_dipole_Boltz", "ligand_molar_volume": "ligand_molar_volume_Boltz",
}  # fmt: skip
BAS = {
    "base_dipole": "base_dipole",
    "base_molar_volume": "base_molar_volume",
    "base_E_HOMO": "base_homo_energy",
    "base_E_LUMO": "base_lumo_energy",
}
SOL = {
    "solvent_dipole": "solvent_dipole",
    "solvent_molar_volume": "solvent_molar_volume",
    "solvent_E_HOMO": "solvent_homo_energy",
    "solvent_E_LUMO": "solvent_lumo_energy",
}

df = idx.merge(
    lig[["ligand_SMILES", "ligand_file_name", *LIG.values()]].rename(
        columns={"ligand_SMILES": "Ligand_SMILES"}
    ),
    on="Ligand_SMILES",
    how="left",
)
df = df.merge(
    bas[["base_SMILES", "base_file_name", *BAS.values()]].rename(
        columns={"base_SMILES": "Base_SMILES"}
    ),
    on="Base_SMILES",
    how="left",
)
df = df.merge(
    sol[["solvent_SMILES", "solvent_file_name", *SOL.values()]].rename(
        columns={"solvent_SMILES": "Solvent_SMILES"}
    ),
    on="Solvent_SMILES",
    how="left",
)
assert df[list(LIG.values())].notna().all().all() and len(df) == len(idx)
df["ligand_pct_Vbur_P"] = 100.0 * df["ligand_atom1_%VBur_Boltz"]  # fraction to percent

X_df = pd.DataFrame(
    {
        "temperature_C": df["Temp_C"].astype(float),
        "concentration_M": df["Concentration"].astype(float),
    }
)
for new, old in {**LIG, **BAS, **SOL}.items():
    if new != "ligand_pct_Vbur_P":
        X_df[new] = df[old].astype(float)
    else:
        X_df[new] = df[new]
COMPONENT = {"temperature_C": "conditions", "concentration_M": "conditions"}
COMPONENT.update({k: "ligand" for k in LIG})
COMPONENT.update({k: "base" for k in BAS})
COMPONENT.update({k: "solvent" for k in SOL})
y = df["yield"].to_numpy(dtype=float)
TASK = "regression"
Y_LABEL = "yield (%)"
SUCCESS = (y > 20.0).astype(int)
SUCCESS_RULE = "yield above 20 %"
print(
    f"{len(df)} reactions: {df.ligand_file_name.nunique()} ligands x {df.base_file_name.nunique()} bases x {df.solvent_file_name.nunique()} solvents x {df.Concentration.nunique()} concentrations x {df.Temp_C.nunique()} temperatures"
)
for comp in ["ligand_file_name", "base_file_name", "solvent_file_name", "Temp_C", "Concentration"]:
    print(df.groupby(comp)["yield"].mean().round(1).sort_values(ascending=False).to_dict())
GROUPS = df["ligand_file_name"].to_numpy()
HYPOTHESIS = []

In [ ]:
shifts = {
    "new ligand": NewGroups(df["ligand_file_name"].to_numpy(), "new ligand"),
    "new base": NewGroups(df["base_file_name"].to_numpy(), "new base"),
    "new solvent": NewGroups(df["solvent_file_name"].to_numpy(), "new solvent"),
    "higher temperature": BeyondRange(
        X_df["temperature_C"].to_numpy(), "high", name="higher temperature"
    ),
}
L = run_dataset(
    "Direct arylation with reaction conditions",
    X_df,
    y,
    shifts,
    family="gaussian",
    component=COMPONENT,
)

## 6. Alcohol deoxyfluorination

Nielsen et al., JACS 2018: 740 reactions over 37 alcohols, 5 sulfonyl fluorides and 4 bases. Each component is held out in turn.

In [ ]:
URL = "https://raw.githubusercontent.com/doyle-lab-ucla/ochem-data/main/deoxyF/rxns/dft_from_paper/all.csv"
raw = pd.read_csv(URL)
X_df = raw.drop(columns=["index", "yield"]).copy()
const = [c for c in X_df.columns if X_df[c].nunique() <= 1]
X_df = X_df.drop(columns=const)
print(f"Dropped constant descriptors: {const}")
COMPONENT = {
    c: (
        "alcohol"
        if c.startswith("alcohol")
        else "base"
        if c.startswith("base")
        else "sulfonyl fluoride"
    )
    for c in X_df.columns
}
y = raw["yield"].to_numpy(dtype=float)
TASK = "regression"
Y_LABEL = "yield (%)"
SUCCESS = (y > 40.0).astype(int)
SUCCESS_RULE = "yield above 40 %"
n_alc = raw[[c for c in raw.columns if c.startswith("alcohol")]].drop_duplicates().shape[0]
print(
    f"{len(raw)} reactions: {n_alc} alcohols x {raw['base_.N1_exposed_area'].nunique()} bases x {raw['sulfonyl_fluoride_.S1_electrostatic_charge'].nunique()} sulfonyl fluorides"
)
cls = np.select(
    [raw.alcohol_primary == 1, raw.alcohol_secondary == 1, raw.alcohol_tertiary == 1],
    ["primary", "secondary", "tertiary"],
    "other",
)
print(
    pd.DataFrame({"class": cls, "yield": y})
    .groupby("class")["yield"]
    .agg(["mean", "count"])
    .round(1)
)
GROUPS = (
    raw[[c for c in raw.columns if c.startswith("alcohol")]]
    .round(6)
    .astype(str)
    .agg("|".join, axis=1)
    .to_numpy()
)
HYPOTHESIS = []

In [ ]:
shifts = {
    "new alcohol": NewGroups(GROUPS, "new alcohol"),
    "new sulfonyl fluoride": NewGroups(
        groups_of(raw, [c for c in raw.columns if c.startswith("sulfonyl")]),
        "new sulfonyl fluoride",
    ),
    "new base": NewGroups(
        groups_of(raw, [c for c in raw.columns if c.startswith("base")]), "new base"
    ),
}
L = run_dataset(
    "Alcohol deoxyfluorination", X_df, y, shifts, family="gaussian", component=COMPONENT
)

## 7. Another field: house prices

Nothing above is specific to chemistry. The California housing table (median house value per
census block, 1990; 4000 blocks sampled) with the same calls and three shifts: a *region* never
seen (blocks clustered by latitude and longitude, whole regions held out), incomes above every
training income, and blocks unlike any trained on (clusters of the other variables).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_california_housing

H = (
    fetch_california_housing(as_frame=True)
    .frame.sample(4000, random_state=0)
    .reset_index(drop=True)
)
region = KMeans(40, n_init=4, random_state=0).fit_predict(H[["Latitude", "Longitude"]])
X_h, y_h = H.drop(columns="MedHouseVal"), H["MedHouseVal"].to_numpy()
shifts = {
    "new region": NewGroups(region, "new region"),
    "higher income": BeyondRange(X_h["MedInc"].to_numpy(), "high", name="higher income"),
    "unlike anything seen": NewGroups.from_clusters(
        X_h.drop(columns=["Latitude", "Longitude"]).to_numpy(), 12, "unlike anything seen"),
}  # fmt: skip
L = run_dataset("House prices", X_h, y_h, shifts, prune=False,
                component={"Latitude": "location", "Longitude": "location"})  # fmt: skip
print("\nThree blocks with an income far above anything fitted:")
print(L.models["equation"].support(X_h.head(3).assign(MedInc=30.0)))

## 8. Across the datasets

One row per dataset and shift: the skill of each rung on the reference and under the shift, the
memorized share, and the verdicts on the descriptor blocks (or on the variables, where no
blocks were given).

In [ ]:
rows = []
for tag, L in LEDGERS.items():
    lad = L.ladder
    for s in L.schemes[1:]:
        row = {"dataset": tag, "shift": s, "units": L.meta["units"][s]}
        for r, short in [("equation", "equation"), (f"equation, chosen for {s}", "equation, chosen for it"),
                         (f"curves, tuned for {s}", "curves, tuned for it"),
                         ("KAN, one hidden layer", "KAN"), ("gradient boosting", "boosting")]:  # fmt: skip
            if r in lad.index:
                row[f"{short}: reference"] = lad.loc[r, "reference"]
                row[f"{short}: shift"] = lad.loc[r, s]
        row["memorized"] = L.memorized.loc[s, "memorized"]
        v = L.block_verdicts[s] if len(L.block_verdicts) else L.verdicts[s]
        for verdict in ["transfers", "does not transfer", "undetermined"]:
            row[verdict] = ", ".join(v.index[v == verdict])
        rows.append(row)
summary = pd.DataFrame(rows).set_index(["dataset", "shift"])
text = ["transfers", "does not transfer", "undetermined"]
display(summary.drop(columns=text).round(2))
display(summary[text])

### Reading

_(written after the run)_